- conv1: 7×7, stride 2, pad 3, 64 channels.
- maxpool: 3×3, stride 2, pad 1.
- layer1: keeps the spatial size, 64 channels.
- layer2, layer3 and layer4: each halves the spatial size, and channels go 128, 256 and 512.

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()
for p in model.parameters():
    p.requires_grad_(False)

In [2]:
feats = {}
def make_hook(name):
    def hook(module, inp, out):
        feats[name] = out.detach()
    return hook

handles = [getattr(model, n).register_forward_hook(make_hook(n))
           for n in ["layer1", "layer2", "layer3", "layer4"]]

x = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    model(x)
print({k: tuple(v.shape) for k, v in feats.items()})

{'layer1': (1, 64, 56, 56), 'layer2': (1, 128, 28, 28), 'layer3': (1, 256, 14, 14), 'layer4': (1, 512, 7, 7)}


In [3]:
with torch.no_grad():
    a = model(x); b = model(x)
print("deterministic:", torch.equal(a, b))

with torch.no_grad():
    model(torch.zeros(1, 3, 448, 448))
print({k: tuple(v.shape) for k, v in feats.items()})

for h in handles:
    h.remove()

deterministic: True
{'layer1': (1, 64, 112, 112), 'layer2': (1, 128, 56, 56), 'layer3': (1, 256, 28, 28), 'layer4': (1, 512, 14, 14)}
